# Chapter 6: Other computer vision problems

In [2]:
! pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.1/124.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.9/246.9 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 45.8 MB/s eta 0:00:00:00:01


In [3]:
from fastai.vision.all import *
from fastbook import *

In [35]:
def list_methods(var):
    return [a for a in dir(var) if callable(getattr(var, a))]

## Part 1: Multi-label data

### Block 1: Multi-label vs. single-label: when labels aren't mutually exclusive

In some datasets, data metadata is stored in a csv file that specifies, among other things, where the actual data files live. The PASCAL dataset falls into this category, and that's what we're using for this chapter.

In [11]:
path = untar_data(URLs.PASCAL_2007)
path

<div><progress max="1637796771" value="1637801984"></progress> 100.00% [1637801984/1637796771 01:55&lt;00:00]</div>

Path('/root/.fastai/data/pascal_2007')

In [12]:
Path.BASE_PATH = path
path.ls()

[Path('test.json'), Path('train.csv'), Path('test'), Path('valid.json'), Path('train'), Path('test.csv'), Path('segmentation'), Path('train.json')]

Notice the `train.csv` and `test.csv` files.

In [13]:
(path/'train').ls()

(#5012) [Path('train/009528.jpg'), Path('train/009721.jpg'), Path('train/006269.jpg'), Path('train/007762.jpg'), Path('train/001586.jpg'), Path('train/004846.jpg'), Path('train/006628.jpg'), Path('train/007899.jpg'), Path('train/001981.jpg'), Path('train/008722.jpg'), Path('train/000030.jpg'), Path('train/004955.jpg'), Path('train/005421.jpg'), Path('train/005230.jpg'), Path('train/007330.jpg'), Path('train/002411.jpg'), Path('train/003658.jpg'), Path('train/002657.jpg'), Path('train/008398.jpg'), Path('train/000777.jpg'), Path('train/004244.jpg'), Path('train/003063.jpg'), Path('train/000009.jpg'), Path('train/009809.jpg'), Path('train/001231.jpg'), Path('train/009654.jpg'), Path('train/006802.jpg'), Path('train/009792.jpg'), Path('train/004185.jpg'), Path('train/008468.jpg'), Path('train/001517.jpg'), Path('train/001521.jpg'), Path('train/004196.jpg'), Path('train/007606.jpg'), Path('train/006983.jpg'), Path('train/009022.jpg'), Path('train/009443.jpg'), Path('train/003859.jpg'), Pat

To better understand the data, we need to look at the contents of the CSV files. For that we'll use pandas.

In [14]:
import pandas as pd

In [49]:
train = pd.read_csv((path/'train.csv'))
train

,fname,labels,is_valid
0,000005.jpg,chair,True
1,000007.jpg,car,True
2,000009.jpg,horse person,True
3,000012.jpg,car,False
4,000016.jpg,bicycle,True
...,...,...,...
5006,009954.jpg,horse person,True
5007,009955.jpg,boat,True
5008,009958.jpg,person bicycle,True
5009,009959.jpg,car,False


The columns are clear:
- `fname` is the filename
- `labels` tells us the labels for the image; we can see that a space-separated list of values indicates that there are multiple appropriate labels for an image
- `is_valid` indicates whether the image should be in the training or validation set

Let's find the rows that have multiple labels.

In [43]:
train[train['labels'].str.contains(' ')]

,fname,labels,is_valid
2,000009.jpg,horse person,True
5,000017.jpg,person horse,False
8,000021.jpg,dog person,True
9,000023.jpg,bicycle person,False
12,000030.jpg,bicycle person,True
...,...,...,...
5003,009947.jpg,boat person,True
5004,009949.jpg,sofa chair person,False
5005,009950.jpg,train person,True
5006,009954.jpg,horse person,True


So a little less than half of the rows have multiple labels.